In [1]:
"""
Dangote Refinery IPO Intelligence - Analytical Engine (v2)
==========================================================
Corrected transaction costs (NGX actual) and cleaner investment logic.

Investment amount = total capital available (costs deducted from this).
Break-even = the exit price at which net proceeds equal total outflow.
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. CONFIGURATION
# ============================================================
EXCEL_FILE = r"C:\Users\USER\Desktop\IPO Project\Dangote_Petroleum_Refinery_IPO_Data-v2.xlsx"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Verified prospectus constants ---
IPO_PRICE         = 525.00
SHARES_OFFERED    = 4_100_000_000
SHARES_PRE_OFFER  = 120_128_915_901
SHARES_POST_OFFER = 124_228_915_901
GROSS_PROCEEDS    = 2_152_500_000_000
OFFER_COSTS       = 41_492_782_688.91
NET_PROCEEDS      = GROSS_PROCEEDS - OFFER_COSTS

# --- H1 2026 income statement (₦'million) ---
H1_REVENUE       = 19_134_942
H1_COGS          = 15_702_296
H1_GROSS_PROFIT  = 3_432_646
H1_OPEX          = 215_983
H1_EBIT          = 3_253_838
H1_D_AND_A       = 324_994
H1_FINANCE_COSTS = 424_765
H1_PBT           = 2_897_271
H1_TAX           = 392_839
H1_PAT           = 2_504_432

# --- Balance sheet (₦'million, 30-Jun-2026) ---
CASH                 = 5_891_559
SHORT_TERM_DEBT      = 2_168_529
LONG_TERM_DEBT       = 5_653_742
TOTAL_EQUITY         = 14_677_634
NET_DEBT             = 1_930_712
NET_DEBT_USD_THOUS   = 1_398_468

FX_RATE = NET_DEBT / NET_DEBT_USD_THOUS   # ≈ ₦1,380.59/$

# --- NGX transaction cost assumptions (verified rates) ---
BUY_COST_RATE  = 0.02235   # brokerage 1.5 + SEC 0.3 + NGX 0.3 + CSCS 0.06 + stamp 0.075
SELL_COST_RATE = 0.02160   # brokerage 1.5 + SEC 0.3 + NGX 0.3 + CSCS 0.06
ROUND_TRIP     = BUY_COST_RATE + SELL_COST_RATE

# --- External benchmarks (analyst estimates, not from prospectus) ---
PEER_MEDIAN_EV_EBITDA_GTI   = 4.79
PEER_MEDIAN_PE_GTI          = 8.54
PEER_MEDIAN_EV_EBITDA_BROAD = 7.52
VALERO_EV_EBITDA            = 8.50
MARATHON_EV_EBITDA          = 8.21

GTI_BEAR = 328
GTI_BASE = 503
GTI_BULL = 640

# ============================================================
# 2. LOAD WORKBOOK (audit trail)
# ============================================================
print(f"Loading {EXCEL_FILE} ...")
try:
    xl = pd.read_excel(EXCEL_FILE, sheet_name=None, header=None)
    print(f"  Sheets loaded: {list(xl.keys())}")
except FileNotFoundError:
    print(f"  WARNING: {EXCEL_FILE} not found in current folder.")
    print("  Continuing with verified constants only.")

# ============================================================
# 3. VALUATION SNAPSHOT
# ============================================================
pat_annualised     = H1_PAT * 2
ebitda_h1          = H1_EBIT + H1_D_AND_A
ebitda_annualised  = ebitda_h1 * 2
revenue_annualised = H1_REVENUE * 2

eps_pre  = (pat_annualised * 1_000_000) / SHARES_PRE_OFFER
eps_post = (pat_annualised * 1_000_000) / SHARES_POST_OFFER

pe_pre   = IPO_PRICE / eps_pre
pe_post  = IPO_PRICE / eps_post

mktcap_pre  = IPO_PRICE * SHARES_PRE_OFFER
mktcap_post = IPO_PRICE * SHARES_POST_OFFER
net_debt_ngn = NET_DEBT * 1_000_000
ev_pre       = mktcap_pre  + net_debt_ngn
ev_post      = mktcap_post + net_debt_ngn
ev_ebitda_pre  = ev_pre  / (ebitda_annualised * 1_000_000)
ev_ebitda_post = ev_post / (ebitda_annualised * 1_000_000)

ps_pre  = mktcap_pre  / (revenue_annualised * 1_000_000)
ps_post = mktcap_post / (revenue_annualised * 1_000_000)

valuation_summary = pd.DataFrame({
    "Metric": [
        "IPO Price (₦)", "Shares Pre-Offer", "Shares Post-Offer",
        "Market Cap Pre-Offer (₦tn)", "Market Cap Post-Offer (₦tn)",
        "Annualised Revenue (₦tn)", "Annualised PAT (₦tn)",
        "Annualised EBITDA (₦tn)", "EPS Pre-Offer (₦)", "EPS Post-Offer (₦)",
        "P/E Pre-Offer (x)", "P/E Post-Offer (x)",
        "Net Debt (₦tn)", "EV Pre-Offer (₦tn)", "EV Post-Offer (₦tn)",
        "EV/EBITDA Pre-Offer (x)", "EV/EBITDA Post-Offer (x)",
        "P/S Pre-Offer (x)", "P/S Post-Offer (x)",
        "FX Rate (₦/$)"
    ],
    "Value": [
        IPO_PRICE, SHARES_PRE_OFFER, SHARES_POST_OFFER,
        round(mktcap_pre / 1e12, 2), round(mktcap_post / 1e12, 2),
        round(revenue_annualised / 1e6, 2), round(pat_annualised / 1e6, 2),
        round(ebitda_annualised / 1e6, 2), round(eps_pre, 2), round(eps_post, 2),
        round(pe_pre, 2), round(pe_post, 2),
        round(net_debt_ngn / 1e12, 2), round(ev_pre / 1e12, 2), round(ev_post / 1e12, 2),
        round(ev_ebitda_pre, 2), round(ev_ebitda_post, 2),
        round(ps_pre, 2), round(ps_post, 2),
        round(FX_RATE, 2)
    ]
})
print("\n--- Valuation Snapshot ---")
print(valuation_summary.to_string(index=False))

# ============================================================
# 4. BREAK-EVEN PRICE (after costs)
# ============================================================
break_even_price = IPO_PRICE * (1 + BUY_COST_RATE) / (1 - SELL_COST_RATE)
break_even_pct   = (break_even_price / IPO_PRICE - 1) * 100

print(f"\n--- Break-Even ---")
print(f"Buy cost rate:  {BUY_COST_RATE*100:.3f}%")
print(f"Sell cost rate: {SELL_COST_RATE*100:.3f}%")
print(f"Round-trip:     {ROUND_TRIP*100:.3f}%")
print(f"Break-even price: ₦{break_even_price:.2f} ({break_even_pct:+.2f}% vs ₦{IPO_PRICE})")

# ============================================================
# 5. PEER COMPARISON
# ============================================================
peer_comparison = pd.DataFrame({
    "Benchmark": [
        "Dangote Refinery (at ₦525)",
        "GTI Peer Median (pure-play refiners)",
        "Businessday Broad Median",
        "Valero Energy",
        "Marathon Petroleum"
    ],
    "EV_EBITDA_x": [
        round(ev_ebitda_post, 2),
        PEER_MEDIAN_EV_EBITDA_GTI,
        PEER_MEDIAN_EV_EBITDA_BROAD,
        VALERO_EV_EBITDA,
        MARATHON_EV_EBITDA
    ],
    "PE_x": [
        round(pe_post, 2),
        PEER_MEDIAN_PE_GTI,
        np.nan, np.nan, np.nan
    ],
    "Source": [
        "Calculated from prospectus",
        "GTI Research (analyst estimate)",
        "Businessday (analyst estimate)",
        "Businessday (analyst estimate)",
        "Businessday (analyst estimate)"
    ]
})
peer_comparison["Dangote_Premium_pct"] = (
    (ev_ebitda_post / peer_comparison["EV_EBITDA_x"] - 1) * 100
).round(1)

print("\n--- Peer Comparison ---")
print(peer_comparison.to_string(index=False))

# ============================================================
# 6. SCENARIO MATRIX
# Investment amount = total capital available (costs deducted from this).
# ============================================================
INVESTMENT_AMOUNTS = [20_000, 50_000, 100_000, 250_000, 500_000, 1_000_000]
EXIT_PRICES        = [300, 400, 500, 525, 548.59, 600, 700, 800, 1000, 1200, 1500, 2000]

rows = []
for capital in INVESTMENT_AMOUNTS:
    # Max shares affordable given buy cost
    shares = int(capital // (IPO_PRICE * (1 + BUY_COST_RATE)))
    if shares == 0:
        continue
    cost_basis  = shares * IPO_PRICE
    buy_cost    = cost_basis * BUY_COST_RATE
    total_out   = cost_basis + buy_cost
    for ep in EXIT_PRICES:
        gross_exit = shares * ep
        sell_cost  = gross_exit * SELL_COST_RATE
        net_exit   = gross_exit - sell_cost
        profit     = net_exit - total_out
        roi        = (profit / total_out) * 100 if total_out > 0 else 0
        rows.append({
            "Capital (₦)":      capital,
            "Exit Price (₦)":   ep,
            "Shares":           shares,
            "Cost Basis (₦)":   round(cost_basis, 2),
            "Buy Cost (₦)":     round(buy_cost, 2),
            "Total Outflow (₦)":round(total_out, 2),
            "Gross Exit (₦)":   round(gross_exit, 2),
            "Sell Cost (₦)":    round(sell_cost, 2),
            "Net Exit (₦)":     round(net_exit, 2),
            "Net Profit (₦)":   round(profit, 2),
            "ROI (%)":          round(roi, 2)
        })

scenario_matrix = pd.DataFrame(rows)
print("\n--- Scenario Matrix (₦20,000 capital) ---")
print(scenario_matrix[scenario_matrix["Capital (₦)"] == 20_000].to_string(index=False))

# ============================================================
# 7. MINIMUM INVESTMENT CALCULATOR
# ============================================================
TARGET_PROFITS = [5_000, 10_000, 25_000, 50_000, 100_000, 250_000]

min_rows = []
for target in TARGET_PROFITS:
    for ep in EXIT_PRICES:
        profit_per_share = ep * (1 - SELL_COST_RATE) - IPO_PRICE * (1 + BUY_COST_RATE)
        if profit_per_share <= 0:
            min_rows.append({
                "Target Profit (₦)": target,
                "Exit Price (₦)": ep,
                "Min Capital (₦)": np.nan,
                "Shares Needed": np.nan,
                "Note": "Below break-even — profit not achievable"
            })
            continue
        shares_needed = int(np.ceil(target / profit_per_share))
        min_capital   = shares_needed * IPO_PRICE * (1 + BUY_COST_RATE)
        min_rows.append({
            "Target Profit (₦)": target,
            "Exit Price (₦)": ep,
            "Min Capital (₦)": round(min_capital, 2),
            "Shares Needed": shares_needed,
            "Note": ""
        })

min_inv_df = pd.DataFrame(min_rows)
print("\n--- Minimum Capital for ₦10,000 Profit ---")
print(min_inv_df[min_inv_df["Target Profit (₦)"] == 10_000].to_string(index=False))

# ============================================================
# 8. GTI FAIR VALUE SCENARIOS
# ============================================================
gti = pd.DataFrame({
    "Scenario": ["Bear", "Base", "Bull"],
    "Fair Value (₦)": [GTI_BEAR, GTI_BASE, GTI_BULL],
    "Source": ["GTI Research", "GTI Research", "GTI Research"]
})
gti["Return vs 525 (%)"] = ((gti["Fair Value (₦)"] / IPO_PRICE - 1) * 100).round(1)

# Profit on ₦20,000 using the same cost model
cap = 20_000
sh  = int(cap // (IPO_PRICE * (1 + BUY_COST_RATE)))
out = sh * IPO_PRICE * (1 + BUY_COST_RATE)
profits = []
for fv in gti["Fair Value (₦)"]:
    net = sh * fv * (1 - SELL_COST_RATE)
    profits.append(round(net - out, 2))
gti["Profit on ₦20,000 (₦)"] = profits

print("\n--- GTI Fair Value Scenarios ---")
print(gti.to_string(index=False))

# ============================================================
# 9. EXPORT CSVs
# ============================================================
print(f"\nExporting CSVs to {OUTPUT_DIR}/ ...")
valuation_summary.to_csv(OUTPUT_DIR / "valuation_summary.csv", index=False)
peer_comparison.to_csv(OUTPUT_DIR / "peer_comparison.csv", index=False)
scenario_matrix.to_csv(OUTPUT_DIR / "scenario_matrix.csv", index=False)
min_inv_df.to_csv(OUTPUT_DIR / "minimum_investment.csv", index=False)
gti.to_csv(OUTPUT_DIR / "gti_scenarios.csv", index=False)

assumptions = pd.DataFrame({
    "Assumption": [
        "FX Rate (₦/$)", "Buy Cost Rate (%)", "Sell Cost Rate (%)",
        "Round-Trip Cost (%)", "Break-Even Price (₦)",
        "Break-Even Move (%)", "Annualisation Method",
        "Holding Period", "Dividends", "Peer Multiples Source"
    ],
    "Value": [
        round(FX_RATE, 2), BUY_COST_RATE*100, SELL_COST_RATE*100,
        ROUND_TRIP*100, round(break_even_price, 2),
        round(break_even_pct, 2), "H1 2026 × 2 (conservative)",
        "12 months from listing (assumption)",
        "Excluded; subject to 10% withholding tax if paid",
        "GTI Research & Businessday (analyst estimates)"
    ]
})
assumptions.to_csv(OUTPUT_DIR / "assumptions.csv", index=False)

print("  Files created:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"    - {f}")

# ============================================================
# 10. KEY TAKEAWAYS
# ============================================================
print("\n=== KEY TAKEAWAYS ===")
print(f"At ₦525, Dangote trades at {pe_post:.1f}x P/E and {ev_ebitda_post:.1f}x EV/EBITDA.")
print(f"GTI peer median: {PEER_MEDIAN_EV_EBITDA_GTI}x EV/EBITDA. Dangote is at a premium.")
print(f"Break-even price: ₦{break_even_price:.2f} ({break_even_pct:.2f}% above IPO).")
row = min_inv_df[(min_inv_df['Target Profit (₦)']==10_000) & (min_inv_df['Exit Price (₦)']==700)]
if not row.empty:
    print(f"Minimum capital for ₦10,000 profit at ₦700 exit: "
          f"₦{row.iloc[0]['Min Capital (₦)']:,.0f} ({int(row.iloc[0]['Shares Needed'])} shares)")

Loading C:\Users\USER\Desktop\IPO Project\Dangote_Petroleum_Refinery_IPO_Data-v2.xlsx ...
  Sheets loaded: ['Executive Summary', 'Offer Terms & Ownership', 'Financial Statements', 'Operational & Proceeds', 'Tax & Valuation', 'Risks & Key Dates']

--- Valuation Snapshot ---
                     Metric        Value
              IPO Price (₦) 5.250000e+02
           Shares Pre-Offer 1.201289e+11
          Shares Post-Offer 1.242289e+11
 Market Cap Pre-Offer (₦tn) 6.307000e+01
Market Cap Post-Offer (₦tn) 6.522000e+01
   Annualised Revenue (₦tn) 3.827000e+01
       Annualised PAT (₦tn) 5.010000e+00
    Annualised EBITDA (₦tn) 7.160000e+00
          EPS Pre-Offer (₦) 4.170000e+01
         EPS Post-Offer (₦) 4.032000e+01
          P/E Pre-Offer (x) 1.259000e+01
         P/E Post-Offer (x) 1.302000e+01
             Net Debt (₦tn) 1.930000e+00
         EV Pre-Offer (₦tn) 6.500000e+01
        EV Post-Offer (₦tn) 6.715000e+01
    EV/EBITDA Pre-Offer (x) 9.080000e+00
   EV/EBITDA Post-Offer (x) 9